In [ ]:
import pandas as pd
import numpy as np
import requests

df = pd.read_parquet('../data/raw/archive.parquet')

In [ ]:
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 1031403 entries, 0 to 1031402
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype              
---  ------       --------------    -----              
 0   login        1031403 non-null  str                
 1   actor_id     1031403 non-null  Int64              
 2   n_events     1031403 non-null  Int64              
 3   n_repos      1031403 non-null  Int64              
 4   active_days  1031403 non-null  Int64              
 5   n_pushes     1031403 non-null  Int64              
 6   n_prs        1031403 non-null  Int64              
 7   first_seen   1031403 non-null  datetime64[us, UTC]
 8   last_seen    1031403 non-null  datetime64[us, UTC]
 9   repos        1031403 non-null  object             
dtypes: Int64(6), datetime64[us, UTC](2), object(1), str(1)
memory usage: 95.6+ MB


In [5]:
df

,login,actor_id,n_events,n_repos,active_days,n_pushes,n_prs,first_seen,last_seen,repos
0,SethMorrowSoftware,140758907,24,4,6,20,3,2026-10-20 21:07:45+00:00,2026-10-26 18:12:45+00:00,"[SethMorrowSoftware/OpenXTalk-Beyond, SethMorr..."
1,brendanjerwin,3039,22,1,5,19,1,2026-10-21 22:26:49+00:00,2026-10-27 15:53:46+00:00,[brendanjerwin/simple_wiki]
2,Neferchipss,80695926,208,3,6,208,0,2026-10-21 06:20:24+00:00,2026-10-26 09:18:06+00:00,"[Neferchipss/ada-live, Neferchipss/babajee, Ne..."
3,avcn101-del,247913993,31,1,6,26,5,2026-10-20 14:23:46+00:00,2026-10-25 14:08:48+00:00,[avcn101-del/Starship_construction]
4,DevelopPioneerNavigate,112266230,27,7,4,27,0,2026-10-22 09:04:19+00:00,2026-10-26 02:46:14+00:00,"[DevelopPioneerNavigate/DPN-PhaseGate, Develop..."
...,...,...,...,...,...,...,...,...,...,...
1031398,kerryhatcher,1011814,13,4,5,10,3,2026-10-20 19:24:17+00:00,2026-10-26 20:08:55+00:00,"[kerryhatcher/www.civpulse.org, kerryhatcher/f..."
1031399,abahmed,16365652,10,2,3,10,0,2026-10-21 15:25:24+00:00,2026-10-25 23:52:40+00:00,"[abahmed/kwatch, abahmed/kwatch.dev]"
1031400,Manish15559,114980892,10,1,4,10,0,2026-10-20 06:19:51+00:00,2026-10-25 18:52:01+00:00,[Manish15559/LeetcodeSolution]
1031401,LeviGuilherme,56275425,10,2,4,10,0,2026-10-20 20:15:21+00:00,2026-10-25 18:13:51+00:00,"[Levix-Digital/gedo, Levix-Digital/any-context]"


In [ ]:
active = df[(df['active_days'] >= 3) & (df['n_events'] >= 5) & (df['n_events'] <= 400)]

In [7]:
active

,login,actor_id,n_events,n_repos,active_days,n_pushes,n_prs,first_seen,last_seen,repos
0,SethMorrowSoftware,140758907,24,4,6,20,3,2026-10-20 21:07:45+00:00,2026-10-26 18:12:45+00:00,"[SethMorrowSoftware/OpenXTalk-Beyond, SethMorr..."
1,brendanjerwin,3039,22,1,5,19,1,2026-10-21 22:26:49+00:00,2026-10-27 15:53:46+00:00,[brendanjerwin/simple_wiki]
2,Neferchipss,80695926,208,3,6,208,0,2026-10-21 06:20:24+00:00,2026-10-26 09:18:06+00:00,"[Neferchipss/ada-live, Neferchipss/babajee, Ne..."
3,avcn101-del,247913993,31,1,6,26,5,2026-10-20 14:23:46+00:00,2026-10-25 14:08:48+00:00,[avcn101-del/Starship_construction]
4,DevelopPioneerNavigate,112266230,27,7,4,27,0,2026-10-22 09:04:19+00:00,2026-10-26 02:46:14+00:00,"[DevelopPioneerNavigate/DPN-PhaseGate, Develop..."
...,...,...,...,...,...,...,...,...,...,...
1031398,kerryhatcher,1011814,13,4,5,10,3,2026-10-20 19:24:17+00:00,2026-10-26 20:08:55+00:00,"[kerryhatcher/www.civpulse.org, kerryhatcher/f..."
1031399,abahmed,16365652,10,2,3,10,0,2026-10-21 15:25:24+00:00,2026-10-25 23:52:40+00:00,"[abahmed/kwatch, abahmed/kwatch.dev]"
1031400,Manish15559,114980892,10,1,4,10,0,2026-10-20 06:19:51+00:00,2026-10-25 18:52:01+00:00,[Manish15559/LeetcodeSolution]
1031401,LeviGuilherme,56275425,10,2,4,10,0,2026-10-20 20:15:21+00:00,2026-10-25 18:13:51+00:00,"[Levix-Digital/gedo, Levix-Digital/any-context]"


In [8]:

active["hash_val"] = pd.util.hash_pandas_object(active["actor_id"], index=False)
active["bucket"] = active["hash_val"] % 1000

conditions = [
    active["bucket"] <= 3, 
    (active["bucket"] > 3) & (active["bucket"] < 100),
    (active["bucket"] > 100) & (active["bucket"] < 900),
    active["bucket"] >= 900, 
]

choices = ["pilot", "main", "reserve", "control"]
active["bucket_cond"] = np.select(conditions, choices, default="unknown")

active


,login,actor_id,n_events,n_repos,active_days,n_pushes,n_prs,first_seen,last_seen,repos,hash_val,bucket,bucket_cond
0,SethMorrowSoftware,140758907,24,4,6,20,3,2026-10-20 21:07:45+00:00,2026-10-26 18:12:45+00:00,"[SethMorrowSoftware/OpenXTalk-Beyond, SethMorr...",17254391384519280815,815,reserve
1,brendanjerwin,3039,22,1,5,19,1,2026-10-21 22:26:49+00:00,2026-10-27 15:53:46+00:00,[brendanjerwin/simple_wiki],4028197899947845318,318,reserve
2,Neferchipss,80695926,208,3,6,208,0,2026-10-21 06:20:24+00:00,2026-10-26 09:18:06+00:00,"[Neferchipss/ada-live, Neferchipss/babajee, Ne...",1726553521311518325,325,reserve
3,avcn101-del,247913993,31,1,6,26,5,2026-10-20 14:23:46+00:00,2026-10-25 14:08:48+00:00,[avcn101-del/Starship_construction],10640670015920768539,539,reserve
4,DevelopPioneerNavigate,112266230,27,7,4,27,0,2026-10-22 09:04:19+00:00,2026-10-26 02:46:14+00:00,"[DevelopPioneerNavigate/DPN-PhaseGate, Develop...",16343064791225646820,820,reserve
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1031398,kerryhatcher,1011814,13,4,5,10,3,2026-10-20 19:24:17+00:00,2026-10-26 20:08:55+00:00,"[kerryhatcher/www.civpulse.org, kerryhatcher/f...",1423254910730089886,886,reserve
1031399,abahmed,16365652,10,2,3,10,0,2026-10-21 15:25:24+00:00,2026-10-25 23:52:40+00:00,"[abahmed/kwatch, abahmed/kwatch.dev]",7914853269999669733,733,reserve
1031400,Manish15559,114980892,10,1,4,10,0,2026-10-20 06:19:51+00:00,2026-10-25 18:52:01+00:00,[Manish15559/LeetcodeSolution],8612332722295682316,316,reserve
1031401,LeviGuilherme,56275425,10,2,4,10,0,2026-10-20 20:15:21+00:00,2026-10-25 18:13:51+00:00,"[Levix-Digital/gedo, Levix-Digital/any-context]",15367578693858690711,711,reserve


In [9]:
pool = active[active['bucket_cond'] == 'main']

In [10]:
pool

,login,actor_id,n_events,n_repos,active_days,n_pushes,n_prs,first_seen,last_seen,repos,hash_val,bucket,bucket_cond
24,scammarano,30610844,27,4,3,25,2,2026-10-20 03:41:58+00:00,2026-10-25 20:22:39+00:00,"[scammarano/digital-signage-android, scammaran...",7690216223764962069,69,main
25,F3NN3X,186862085,24,2,7,19,5,2026-10-20 01:56:31+00:00,2026-10-27 01:03:29+00:00,"[F3NN3X/InfoPanel.Studio, F3NN3X/InfoPanel.Str...",2296307943267247077,77,main
31,dfed,139364,18,2,6,14,4,2026-10-21 15:09:44+00:00,2026-10-27 23:13:19+00:00,"[dfed/SafeDI, bazelbuild/bazel-central-registry]",12434336129283400076,76,main
36,ShamanOrWhitch,131708486,39,1,3,38,1,2026-10-20 07:37:49+00:00,2026-10-25 22:49:32+00:00,[ShamanOrWhitch/BCM4StarAtlas],1367425805286341093,93,main
71,nickspiker,43169440,31,5,6,31,0,2026-10-20 00:16:42+00:00,2026-10-26 04:09:37+00:00,"[nickspiker/photon, nickspiker/mahere, nickspi...",7502235357906092082,82,main
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1031283,mathiasrw,1063454,10,2,4,10,0,2026-10-20 05:07:22+00:00,2026-10-25 22:28:47+00:00,"[AlaSQL/alasql, mathiasrw/rexreplace]",1624352312553715067,67,main
1031308,HKillua,94744585,10,1,5,10,0,2026-10-20 07:11:21+00:00,2026-10-26 09:12:16+00:00,[HKillua/association-code],9994017623280326053,53,main
1031368,jugglingboss,40720328,10,1,6,10,0,2026-10-20 12:04:47+00:00,2026-10-26 05:01:53+00:00,[jugglingboss/MapTapScores],6828293060355106064,64,main
1031371,Marix201,165239184,10,2,4,10,0,2026-10-23 16:19:04+00:00,2026-10-26 03:43:16+00:00,[CampusSeguro-DOSW-2026-2/ms02-campus-and-secu...,5844878765331124048,48,main


In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
profiles = pd.read_parquet(ROOT / "data" / "processed" / "profiles.parquet")

# 1) First, why are they unclassified? Few usable repos, or a missing vocabulary?
unclassified = profiles[profiles["label"] == "unclassified"]
print(unclassified["n_repos_used"].describe())

# 2) The interesting group: plenty of usable repos, yet scored nothing
suspects = unclassified[unclassified["n_repos_used"] >= 3]
print(len(suspects), "unclassified developers with 3+ usable repos")
sample = suspects.sample(min(30, len(suspects)), random_state=1)

count    186.000000
mean       3.129032
std        3.475710
min        0.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       18.000000
Name: n_repos_used, dtype: float64
75 unclassified developers with 3+ usable repos


In [6]:
df = pd.read_parquet('../data/processed/documents.parquet')

In [7]:
df

,actor_id,login,label,n_repos_used,n_repos_in_doc,n_words,github_url,text
0,262129128,maxd4,frontend,2,2,52,https://github.com/maxd4,"Main languages: TypeScript, JavaScript.\nRepos..."
1,2795616,marmos91,frontend,29,8,152,https://github.com/marmos91,"Main languages: TypeScript, Go, JavaScript, C+..."
2,323695,andrewdunndev,devops_cloud,23,8,257,https://github.com/andrewdunndev,"Main languages: Go, Python, Rust, TypeScript.\..."
3,272655655,alpizar269708-crypto,automation_bots,4,4,24,https://github.com/alpizar269708-crypto,"Main languages: JavaScript, Python.\nRepositor..."
4,75772122,leonaburime-ucla,ml_ai,5,5,53,https://github.com/leonaburime-ucla,"Main languages: JavaScript, TypeScript.\nRepos..."
...,...,...,...,...,...,...,...,...
1793,210417802,Oussamabouhali7,data_science,14,8,233,https://github.com/Oussamabouhali7,"Main languages: Jupyter Notebook, Python, HTML..."
1794,16284436,shashidaren,automation_bots,8,8,43,https://github.com/shashidaren,"Main languages: Python, HTML, TypeScript, Jinj..."
1795,342381956,dineshaan-strangle,automation_bots,6,6,25,https://github.com/dineshaan-strangle,Main languages: Python.\nRepositories:\n- Tren...
1796,55921894,MrAlders0n,automation_bots,6,6,74,https://github.com/MrAlders0n,"Main languages: Python, JavaScript, HTML.\nRep..."
